# BERTopic: Topic Modelling on 10-K MD&A Sentences

**Pipeline**

1. GPU setup
2. Load sentence-level data from `Data Preparation/mda_processed_sample.xlsx` and visualise (head 30 rows)
3. Embed sentences with `all-MiniLM-L6-v2`
4. Fit an **initial BERTopic** model — topic names, intertopic distance map, term score bar chart
5. **Coherence vs K** — sweep `min_cluster_size` to find the optimal number of topics
6. Fit the **final model** with the best K + **KeyBERTInspired** representation
7. **Coherence assessment** and **topic diversity** metrics


In [ ]:
import getpass

# # Ask once, so the token is not stored in plain text in the notebook
GH_TOKEN = getpass.getpass("GitHub token: ")

In [ ]:
!git clone https://$GH_TOKEN@github.com/joyceetran/textmining_grp6.git

## 1. GPU Setup


In [ ]:
# cuML accelerator — transparently replaces sklearn/UMAP/HDBSCAN with GPU versions.
# Must be loaded BEFORE any sklearn/umap/hdbscan imports.
%load_ext cuml.accel

In [ ]:
import torch

# Detect best available device: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    print(f"✅ Using device : {DEVICE}")
    print(f"   GPU count    : {torch.cuda.device_count()}")
    print(f"   GPU name     : {torch.cuda.get_device_name(0)}")
    print(
        f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
elif DEVICE == "mps":
    print("✅ Apple M-series GPU detected — PyTorch MPS backend active.")
else:
    print("⚠️  No GPU detected — running on CPU (will be slow for large datasets).")


In [ ]:
# GPU diagnostics (NVIDIA only)
import subprocess, shutil

if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)


## 2. Imports & Setup


In [ ]:
# Install BERTopic and dependencies (run once — ~2 min on Colab)
!pip install bertopic sentence-transformers umap-learn hdbscan gensim wandb -q

# cuML: GPU-accelerated UMAP + HDBSCAN (RAPIDS — requires CUDA)
# Uses the correct package for your CUDA major version (e.g. cuml-cu12 for CUDA 12.x).
import subprocess, sys, importlib

try:
    import cuml

    print(f"✅ cuML {cuml.__version__} already installed — GPU UMAP/HDBSCAN ready.")
except ImportError:
    import torch

    cuda_ver = torch.version.cuda  # e.g. "12.1"
    if cuda_ver:
        major = cuda_ver.split(".")[0]
        pkg = f"cuml-cu{major}"
        print(f"Installing {pkg} for CUDA {cuda_ver} (may take ~3 min) ...")
        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                pkg,
                "--extra-index-url",
                "https://pypi.nvidia.com",
                "-q",
            ],
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            print(f"✅ {pkg} installed successfully.")
            print("   ⚠️  Restart the Colab runtime now (Runtime → Restart runtime),")
            print("   then re-run from this cell onwards to activate cuML.")
        else:
            print(f"❌ {pkg} install failed — falling back to CPU UMAP/HDBSCAN.")
            print(result.stderr[-500:])
    else:
        print("⚠️  No CUDA detected — cuML skipped; CPU UMAP/HDBSCAN will be used.")


In [ ]:
import warnings, random, time
import os

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from IPython.display import display

# BERTopic stack
from sentence_transformers import SentenceTransformer
from umap import UMAP as CpuUMAP
from hdbscan import HDBSCAN as CpuHDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator
# import wandb  # Temporarily disabled

# cuML: GPU-accelerated UMAP + HDBSCAN (graceful fallback to CPU if unavailable)
try:
    from cuml.manifold import UMAP as GpuUMAP
    from cuml.cluster import HDBSCAN as GpuHDBSCAN

    USE_CUML = True
    print("cuML detected - UMAP and HDBSCAN will run on GPU.")
except ImportError:
    GpuUMAP = CpuUMAP
    GpuHDBSCAN = CpuHDBSCAN
    USE_CUML = False
    print("cuML not available - falling back to CPU UMAP / HDBSCAN.")

# Coherence scoring
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

plt.rcParams["figure.dpi"] = 110
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


class WandbLogger:
    """W&B disabled placeholder to keep notebook flow unchanged."""

    def __init__(self, project, run_name=None, api_key=None):
        self.run = None
        # if api_key:
        #     wandb.login(key=api_key, relogin=True)
        # self.run = wandb.init(project=project, name=run_name)

    def log_hparams(self, params):
        # if self.run is not None:
        #     self.run.config.update(params, allow_val_change=True)
        pass

    def log_metrics(self, metrics):
        # wandb.log(metrics)
        pass

    def log_coherence(self, stage, score, elapsed_sec=None, **extra):
        # payload = {f"coherence/{stage}": float(score)}
        # if elapsed_sec is not None:
        #     payload[f"coherence/{stage}_time_sec"] = float(elapsed_sec)
        # for k, v in extra.items():
        #     payload[f"coherence/{stage}_{k}"] = v
        # wandb.log(payload)
        pass

    def finish(self):
        # wandb.finish()
        pass


WB_PROJECT = "text-mining"
WB_RUN_NAME = "bertopic-run"
WB_API_KEY = os.getenv(
    "WANDB_API_KEY", ""
)  # Preferred: set env var instead of hardcoding.

wb = WandbLogger(
    project=WB_PROJECT,
    run_name=WB_RUN_NAME,
    api_key=WB_API_KEY if WB_API_KEY else None,
)

print("✅ All imports OK")
print("ℹ️ W&B logging is temporarily disabled.")

In [ ]:
# ==================================================================
# RUN 2 HYPERPARAMETERS  (fixed - no sweep needed)
# ==================================================================
MIN_CLUSTER_SIZE = 50  # HDBSCAN - min sentences per topic
MIN_SAMPLES = 15  # HDBSCAN - controls cluster density
MIN_DF = 5  # CountVectorizer - min doc-frequency
NGRAM_RANGE = (1, 3)  # CountVectorizer - up to trigrams
BEST_MCS = MIN_CLUSTER_SIZE  # used by final model directly

print(f"Config  min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}")
print(f"        min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")

# wb.log_hparams(
#     {
#         "min_cluster_size": MIN_CLUSTER_SIZE,
#         "min_samples": MIN_SAMPLES,
#         "min_df": MIN_DF,
#         "ngram_low": NGRAM_RANGE[0],
#         "ngram_high": NGRAM_RANGE[1],
#         "seed": SEED,
#         "device": DEVICE,
#         "use_cuml": USE_CUML,
#     }
# )
print("ℹ️ W&B hyperparameter logging is disabled.")

## 3. Data Loading & Preparation

Load `mda_processed_sample.xlsx` (wide format: 1 row per filing, columns `sent_1…sent_N`)
and melt into a sentence-level long DataFrame used throughout the notebook.


In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

# In Colab the clone lands at /content/textmining_grp6; cd into it so
# all relative paths (datasets/, etc.) work identically to a local run.
_repo = Path("/content/textmining_grp6")
if _repo.exists() and Path.cwd() != _repo:
    os.chdir(_repo)
    print(f"✅ Working directory set to: {_repo}")

DATA_PATH = Path("datasets/final/mda_shared_preprocessed.csv")

if not DATA_PATH.exists():
    print(f"❌ Data file not found: {DATA_PATH.resolve()}")
    raise FileNotFoundError(str(DATA_PATH))

print(f"✅ Loading: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)
print(
    f"✅ Loaded {len(df):,} rows | {df['doc_id'].nunique():,} docs | {df['company_name'].nunique():,} companies"
)
display(df.head(5))


In [ ]:
# Convert sentence-level rows to chunk-level rows
# ─────────────────────────────────────────────────────────────────────────
# WHY CHUNK-LEVEL (not sentence or document level)?
#   • Sentence-level: single sentences (~183 chars) lack enough context for
#     BERTopic — phrases like 'actual results may differ' carry no topic signal.
#   • Document-level: one MD&A mixes liquidity, operations, and risk; topics blur.
#   • Chunk-level (8 sentences ≈ ~1 500 chars): enough context per unit so
#     UMAP+HDBSCAN can find tight, coherent clusters. One doc contributes
#     multiple chunks, so sub-themes within an MD&A are still separable.
# ─────────────────────────────────────────────────────────────────────────

CHUNK_SIZE = 8  # sentences per chunk — 7-10 is optimal for MD&A prose

META_COLS = ["doc_id", "company_name", "filing_type", "filing_date", "year", "quarter"]

# Assign a stable sentence index within each document, then floor-divide into chunks.
df["sent_idx"] = df.groupby("doc_id").cumcount()
df["chunk_idx"] = df["sent_idx"] // CHUNK_SIZE

# Aggregate: join sentences; keep all metadata columns via first() (identical per doc).
df = df.groupby(META_COLS + ["chunk_idx"], as_index=False, sort=False).agg(
    processed_text=("sentence", " ".join)
)
df["processed_text"] = (
    df["processed_text"].str.replace(r"\s+", " ", regex=True).str.strip()
)

docs = df["processed_text"].tolist()

print(f"=== Chunk summary ({CHUNK_SIZE} sentences per chunk) ===")
print(f"Total chunks : {len(df):,}")
print(f"Avg chunk len: {df['processed_text'].str.len().mean():.0f} chars")
display(df.head(30))


## 4. Sentence Embeddings


In [ ]:
EMBED_MODEL = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL} on device={DEVICE} ...")
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

BATCH_SIZE = 1024 if DEVICE == "cuda" else (256 if DEVICE == "mps" else 64)
print(f"Batch size : {BATCH_SIZE}  (device={DEVICE})")

_t0 = time.perf_counter()
embeddings = embed_model.encode(
    docs,
    show_progress_bar=True,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,  # unit-length → cosine ≡ dot, compatible with cuML UMAP
    convert_to_numpy=True,
)
# cuML UMAP expects float32 input
if embeddings.dtype != "float32":
    import numpy as np

    embeddings = embeddings.astype(np.float32)

_embed_sec = time.perf_counter() - _t0
print(f"✅ Embeddings shape : {embeddings.shape}  |  dtype: {embeddings.dtype}")
print(f"   Embedding time   : {_embed_sec:.1f}s")

In [ ]:
# Pre-compute UMAP reduction ONCE — reused by every model fit and the sweep.
# cuML UMAP (GPU): ~10-20s on RTX 3090, no PCA init needed.
# CPU UMAP fallback (MPS machines): PCA warm-start for 3-5x faster convergence.
#
# MPS NOTE: PyTorch MPS is used for *embeddings* only. UMAP/HDBSCAN have no MPS
# backend and always run on CPU. Keep n_components=5 and n_neighbors=15:
#   • n_components=10 was unnecessarily high — standard BERTopic uses 5, and
#     HDBSCAN produces tighter clusters in lower-dimensional space.
#   • n_neighbors=50 on CPU with 50k+ chunks is very slow; 15 is the default
#     and gives better local structure for short financial text chunks.

_t1 = time.perf_counter()

if USE_CUML:
    print("Fitting UMAP on GPU (cuML) ...")
    _umap_base = GpuUMAP(
        n_components=5,
        n_neighbors=15,
        min_dist=0.0,
        metric="cosine",
        random_state=SEED,
    )
else:
    print("Fitting UMAP on CPU (PCA-initialised) ...")

    def _rescale(x):
        x = np.array(x, copy=True).astype(np.float32)
        x /= np.std(x[:, 0]) * 10_000
        return x

    _pca_init = _rescale(
        PCA(n_components=5, random_state=SEED).fit_transform(embeddings)
    )
    _umap_base = CpuUMAP(
        n_components=5,  # lowered from 10 — standard BERTopic default
        n_neighbors=15,  # lowered from 50 — much faster on CPU, better local structure
        min_dist=0.0,
        metric="cosine",
        init=_pca_init,
        low_memory=True,  # trades ~10% speed for lower RAM on large chunk sets
        random_state=SEED,
    )

umap_embeddings = _umap_base.fit_transform(embeddings)
_umap_sec = time.perf_counter() - _t1
backend = "GPU cuML" if USE_CUML else "CPU PCA-init"
print(
    f"✅ UMAP shape : {umap_embeddings.shape}  |  time: {_umap_sec:.1f}s  ({backend})"
)


class PassthroughUMAP(BaseEstimator):
    """Returns pre-computed UMAP embeddings — skips re-fitting for all downstream calls."""

    def __init__(self, precomputed):
        self.precomputed = precomputed

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.precomputed

    def fit_transform(self, X, y=None):
        return self.precomputed


print("✅ PassthroughUMAP ready.")


In [ ]:
# 2-D UMAP for visualize_topics() — fitted on a 5 000-point subsample
# of the original embeddings so topic centroids (384-dim) can be transformed.
# PassthroughUMAP only returns precomputed 5-D doc embeddings and cannot
# project new points (topic centroids), so we keep a separate viz reducer.
_rng = np.random.default_rng(SEED)
_sample_idx = _rng.choice(
    len(embeddings), size=min(5_000, len(embeddings)), replace=False
)
_viz_umap = CpuUMAP(n_components=2, n_neighbors=15, metric="cosine", random_state=SEED)
_viz_umap.fit(embeddings[_sample_idx])
print("✅ 2-D viz UMAP ready.")


## 5. Initial BERTopic Model


In [ ]:
umap_model = PassthroughUMAP(umap_embeddings)

if USE_CUML:
    hdbscan_model = GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_model = CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t3 = time.perf_counter()
topics, probs = topic_model.fit_transform(docs, embeddings)
_initial_sec = time.perf_counter() - _t3
print(f"✅ Initial model fit time: {_initial_sec:.1f}s")

# Reassign outliers to nearest topic by embedding similarity
# Must use original 384-dim embeddings (not umap_embeddings) to match topic_embeddings_ dimensionality
topics = topic_model.reduce_outliers(
    docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0.0
)
topic_model.update_topics(docs, topics=topics)

n_topics = len(set(t for t in topics if t != -1))
n_outliers = sum(1 for t in topics if t == -1)
print(f"   Topics found : {n_topics}")
print(f"   Outliers     : {n_outliers:,} ({n_outliers / len(docs):.1%})")

# wb.log_metrics(
#     {
#         "initial_fit_time_sec": float(_initial_sec),
#         "initial_topics_found": int(n_topics),
#         "initial_outliers": int(n_outliers),
#         "initial_outlier_ratio": float(n_outliers / len(docs)),
#     }
# )
print("ℹ️ W&B metrics logging is disabled for initial model fit.")

In [ ]:
# -- Coherence helper (Gensim Cv) - used throughout the notebook ----------------
import os

_N_WORKERS = max(1, os.cpu_count() - 1)

# Pre-build tokenised corpus + dictionary once - reused by every coherence call
_tokenised = [d.lower().split() for d in docs]
_dictionary = Dictionary(_tokenised)


def compute_coherence(model, stage=None, extra_log=None):
    """Compute Cv coherence; W&B logging is temporarily disabled."""
    t0 = time.perf_counter()
    topic_words = [
        [w for w, _ in words if w and " " not in w]  # skip ngrams not in unigram corpus
        for tid, words in model.get_topics().items()
        if tid != -1
    ]

    if not topic_words:
        score = 0.0
    else:
        try:
            cm = CoherenceModel(
                topics=topic_words,
                texts=_tokenised,
                dictionary=_dictionary,
                coherence="c_v",
                processes=_N_WORKERS,
            )
            score = float(cm.get_coherence())
        except Exception:
            score = 0.0

    elapsed = time.perf_counter() - t0

    # if stage is not None and "wb" in globals():
    #     payload = extra_log.copy() if isinstance(extra_log, dict) else {}
    #     wb.log_coherence(
    #         stage=stage,
    #         score=score,
    #         elapsed_sec=elapsed,
    #         **payload,
    #     )

    return score, elapsed

In [ ]:
# ── Topic Names via get_topic_info() ─────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print("=== Topic Overview ===")
display(topic_info)

print("\n=== Topic Names ===")
for _, row in topic_info.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

## 6. Visualisations — Initial Model

### Intertopic Distance Map

Each bubble represents one topic; bubble **size** is proportional to the number of
documents assigned; **distance** approximates semantic difference. Colours distinguish topics.

### Term Score Bar Chart

Bar height = c-TF-IDF score — how distinctive a term is to its topic vs. all others.


In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"  # Fix rendering in VSCode / Google Colab

# ── Intertopic Distance Map (with topic colours) ───────────────────────────────
# Temporarily swap in the 2-D viz UMAP so BERTopic can project topic centroids.
_saved_umap = topic_model.umap_model
topic_model.umap_model = _viz_umap
fig_idm = topic_model.visualize_topics()
topic_model.umap_model = _saved_umap  # restore PassthroughUMAP
fig_idm.update_layout(
    title_text="Intertopic Distance Map — Initial Model",
    height=600,
)
fig_idm.show()


In [ ]:
# ── Term Score Bar Charts ─────────────────────────────────────────────────────
n_show = min(n_topics, 9)  # cap at 9 for readability
fig_bc = topic_model.visualize_barchart(top_n_topics=n_show, n_words=8)
fig_bc.update_layout(
    title_text="Top Terms per Topic — c-TF-IDF Scores (Initial Model)",
    height=700,
)
fig_bc.show()

## 7. Coherence Validation (Run 2 Params — No Sweep)

Hyperparameters are fixed to **Run 2** values (`min_cluster_size=150`, `min_samples=15`,
`min_df=5`, `ngram_range=(1,3)`). We compute a single coherence score to validate
the configuration rather than sweeping across many values.

To re-enable the full sweep, change the config cell above and replace this section.


In [ ]:
# Single coherence check for Run 2 params - no sweep loop needed.
cv_run2, _cv_sec = compute_coherence(
    topic_model,
    stage="run2",
    extra_log={
        "topics": int(n_topics),
        "min_cluster_size": int(MIN_CLUSTER_SIZE),
        "min_samples": int(MIN_SAMPLES),
    },
)

k_results = [{"mcs": MIN_CLUSTER_SIZE, "k": n_topics, "cv": cv_run2}]
best_k_row = k_results[0]

print(
    f"{'min_cluster_size':>18} {'min_samples':>12} {'k (topics)':>12} {'Coherence Cv':>14}"
)
print("-" * 62)
print(f"{MIN_CLUSTER_SIZE:>18} {MIN_SAMPLES:>12} {n_topics:>12} {cv_run2:>14.4f}")
print(f"\nCoherence computed in {_cv_sec:.1f}s")
print("ℹ️ W&B coherence logging is disabled.")

In [ ]:
# Simple coherence summary bar for the single Run-2 configuration
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(["Run 2"], [cv_run2], color="#2980b9", height=0.4)
ax.axvline(0.55, color="#27ae60", linestyle="--", lw=1.2, label="Good (0.55)")
ax.axvline(0.70, color="#e74c3c", linestyle="--", lw=1.2, label="Very Good (0.70)")
ax.set_xlim(0, max(1.0, cv_run2 + 0.1))
ax.set_xlabel("Coherence Cv", fontsize=11)
ax.set_title(
    f"Run 2 — Cv={cv_run2:.4f}  |  k={n_topics} topics", fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 8. Final BERTopic Model with KeyBERT Representation

We fit the final model using the optimal `min_cluster_size` found above and add
**KeyBERTInspired** as the representation model to produce more semantically precise,
human-readable topic labels than raw c-TF-IDF alone.


In [ ]:
print(f"Final model — min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}")
print(f"             min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")
print("Representation: KeyBERTInspired")
print("-" * 50)

if USE_CUML:
    hdbscan_final = GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_final = CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )

vectorizer_final = CountVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf_final = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

topic_model_final = BERTopic(
    embedding_model=embed_model,
    umap_model=PassthroughUMAP(umap_embeddings),
    hdbscan_model=hdbscan_final,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    representation_model=representation_model,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t4 = time.perf_counter()
topics_final, probs_final = topic_model_final.fit_transform(docs, embeddings)
_final_sec = time.perf_counter() - _t4
print(f"✅ Final model fit time: {_final_sec:.1f}s")

# Reassign outliers to nearest topic by embedding similarity
# Must use original 384-dim embeddings (not umap_embeddings) to match topic_embeddings_ dimensionality
topics_final = topic_model_final.reduce_outliers(
    docs, topics_final, strategy="embeddings", embeddings=embeddings, threshold=0.0
)
topic_model_final.update_topics(docs, topics=topics_final)

n_topics_final = len(set(t for t in topics_final if t != -1))
n_outliers_final = sum(1 for t in topics_final if t == -1)
print(f"   Final Topics  : {n_topics_final}")
print(f"   Final Outliers: {n_outliers_final:,} ({n_outliers_final / len(docs):.1%})")

In [ ]:
# ── Final topic names via get_topic_info() → Name column ─────────────────────
topic_info_final = topic_model_final.get_topic_info()
print("=== Final Topic Overview (KeyBERT-enhanced) ===")
display(topic_info_final)

print("\n=== Final Topic Names ===")
for _, row in topic_info_final.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"  # Fix rendering in VSCode / Google Colab

# ── Final Intertopic Distance Map (with topic colours) ────────────────────────
_saved_umap_final = topic_model_final.umap_model
topic_model_final.umap_model = _viz_umap
fig_final_idm = topic_model_final.visualize_topics()
topic_model_final.umap_model = _saved_umap_final
fig_final_idm.update_layout(
    title_text="Final Model — Intertopic Distance Map",
    height=600,
)
fig_final_idm.show()


In [ ]:
# ── Final Term Score Bar Charts ───────────────────────────────────────────────
n_show_final = min(n_topics_final, 9)
fig_final_bc = topic_model_final.visualize_barchart(
    top_n_topics=n_show_final, n_words=8
)
fig_final_bc.update_layout(
    title_text="Final Model — Top Terms per Topic (KeyBERT Representation)",
    height=700,
)
fig_final_bc.show()

## 9. Coherence Assessment

**Cv coherence** measures how semantically similar the top words within each topic are —
higher is better.

| Cv range  | Interpretation |
| --------- | -------------- |
| < 0.45    | Poor           |
| 0.45–0.55 | Marginal       |
| 0.55–0.70 | Good           |
| > 0.70    | Very good      |


In [ ]:
cv_initial, cv_initial_sec = compute_coherence(
    topic_model,
    stage="initial",
    extra_log={"topics": int(n_topics)},
)
cv_final, cv_final_sec = compute_coherence(
    topic_model_final,
    stage="final",
    extra_log={"topics": int(n_topics_final)},
)

# wb.log_metrics(
#     {
#         "coherence/improvement": float(cv_final - cv_initial),
#         "coherence/initial_time_sec": float(cv_initial_sec),
#         "coherence/final_time_sec": float(cv_final_sec),
#     }
# )

print("=" * 55)
print("COHERENCE ASSESSMENT")
print("=" * 55)
print(f"\n  Initial model Cv : {cv_initial:.4f}")
print(f"  Final model Cv   : {cv_final:.4f}")
print(f"  Improvement      : {cv_final - cv_initial:+.4f}")
print("ℹ️ W&B coherence logging is disabled.")

print()
for score, label in [(cv_initial, "Initial model"), (cv_final, "Final model")]:
    if score >= 0.70:
        msg = "VERY GOOD (>=0.70)"
    elif score >= 0.55:
        msg = "GOOD (0.55-0.70)"
    elif score >= 0.45:
        msg = "MARGINAL (0.45-0.55) - consider tuning"
    else:
        msg = "POOR (<0.45)"
    print(f"  {label:<16}: Cv = {score:.4f}  =>  {msg}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    ["Initial", "Final"],
    [cv_initial, cv_final],
    color=["#7fb3d3", "#2980b9"],
    edgecolor="white",
    width=0.4,
)
for bar, val in zip(bars, [cv_initial, cv_final]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.003,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_ylabel("Coherence Cv")
ax.set_title("Initial vs Final Model - Coherence", fontweight="bold")
ax.set_ylim(0, max(cv_initial, cv_final) * 1.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Topic Diversity

**Topic Diversity** = unique words across all topics ÷ total word slots across all topics.

A high score (close to 1.0) means topics use largely distinct vocabulary — good separation.
A low score means topics share many words — possible redundancy or over-splitting.


In [ ]:
# ── Topic Diversity for final model ──────────────────────────────────────────
all_topic_words = []
for tid, words in topic_model_final.get_topics().items():
    if tid != -1:
        all_topic_words.extend([w for w, _ in words if w])

unique_words = set(all_topic_words)
topic_diversity = len(unique_words) / len(all_topic_words) if all_topic_words else 0.0

print("=" * 55)
print("TOPIC DIVERSITY — Final Model")
print("=" * 55)
print(f"\n  Total word slots : {len(all_topic_words)}")
print(f"  Unique words     : {len(unique_words)}")
print(f"  Topic Diversity  : {topic_diversity:.4f}  ({topic_diversity:.1%})")

if topic_diversity >= 0.70:
    div_msg = "HIGH — topics are well-differentiated"
elif topic_diversity >= 0.50:
    div_msg = "MODERATE — some vocabulary overlap between topics"
else:
    div_msg = "LOW — topics share much vocabulary; consider increasing k"
print(f"\n  Interpretation: {div_msg}")

# Visualise word frequency across topics
from collections import Counter

word_freq = Counter(all_topic_words)
top_shared = [(w, c) for w, c in word_freq.most_common(20) if c > 1]

if top_shared:
    words_s, counts_s = zip(*top_shared)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    colors = ["#e74c3c" if c >= 3 else "#e67e22" for c in counts_s]
    ax.barh(words_s, counts_s, color=colors, edgecolor="white")
    ax.invert_yaxis()
    ax.set_xlabel("Number of topics containing this word")
    ax.set_title(
        f"Words Shared Across Topics (Diversity = {topic_diversity:.1%})",
        fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n(Red bars = word appears in 3+ topics — high overlap indicator)")

## 11. Outlier Experimentation

Run this section if the final model has too many outliers (`Topic = -1`).  
Grid-searches `min_cluster_size` values and tries BERTopic's built-in
`reduce_outliers()` strategies to find the best balance of topic count vs outlier rate.


In [ ]:
# ── 11.1 Current outlier rate ────────────────────────────────────────────────
n_outliers_now = sum(1 for t in topics_final if t == -1)
n_total_now = len(topics_final)
print(
    f"Current outliers : {n_outliers_now:,} / {n_total_now:,} "
    f"({100 * n_outliers_now / n_total_now:.1f}%)"
)

# ── 11.2 Grid-search min_cluster_size ────────────────────────────────────────
# Lower values → more topics, fewer outliers; higher → fewer, broader topics.
mcs_candidates = [20, 30, 40, 50, 75, 100]
exp_results = []

print("\nRunning grid search ...")
for mcs in mcs_candidates:
    ms_exp = max(5, mcs // 5)
    if USE_CUML:
        hdb_exp = GpuHDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms_exp,
            metric="euclidean",
            prediction_data=True,
        )
    else:
        hdb_exp = CpuHDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms_exp,
            metric="euclidean",
            prediction_data=True,
        )

    model_exp = BERTopic(
        embedding_model=embed_model,
        umap_model=PassthroughUMAP(umap_embeddings),
        hdbscan_model=hdb_exp,
        vectorizer_model=CountVectorizer(
            stop_words="english", max_df=0.95, min_df=MIN_DF, ngram_range=NGRAM_RANGE
        ),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        calculate_probabilities=False,
        verbose=False,
    )
    topics_exp, _ = model_exp.fit_transform(docs, embeddings)
    # Reduce outliers with embedding similarity (threshold=0.1 keeps some as -1)
    topics_red = model_exp.reduce_outliers(
        docs, topics_exp, strategy="embeddings", embeddings=embeddings, threshold=0.1
    )
    n_out = sum(1 for t in topics_red if t == -1)
    n_top = len(set(t for t in topics_red if t != -1))
    exp_results.append(
        {
            "mcs": mcs,
            "n_topics": n_top,
            "n_outliers": n_out,
            "outlier_pct": 100 * n_out / n_total_now,
        }
    )
    print(
        f"  mcs={mcs:>3} | {n_top:>3} topics | "
        f"{n_out:>6,} outliers ({100 * n_out / n_total_now:.1f}%)"
    )

# ── 11.3 Visualise trade-off ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
mcs_vals = [r["mcs"] for r in exp_results]
ax1.plot(mcs_vals, [r["outlier_pct"] for r in exp_results], "o-", color="#e74c3c")
ax1.axhline(y=5, color="gray", linestyle="--", label="5% threshold")
ax1.set_xlabel("min_cluster_size")
ax1.set_ylabel("Outlier %")
ax1.set_title("Outlier Rate vs min_cluster_size")
ax1.legend()
ax2.plot(mcs_vals, [r["n_topics"] for r in exp_results], "s-", color="#3498db")
ax2.set_xlabel("min_cluster_size")
ax2.set_ylabel("Number of Topics")
ax2.set_title("Topic Count vs min_cluster_size")
plt.tight_layout()
plt.show()

# ── 11.4 Recommendation ───────────────────────────────────────────────────────
best_exp = min(exp_results, key=lambda r: r["outlier_pct"])
print(
    f"\nRecommended: min_cluster_size={best_exp['mcs']} "
    f"({best_exp['n_topics']} topics, {best_exp['outlier_pct']:.1f}% outliers)"
)
print("Update MIN_CLUSTER_SIZE in run2_config cell and re-run Sections 5 & 8.")


## 12. Topic Labeling

Inspect auto-generated topic keywords and assign human-readable labels.  
Edit `MANUAL_TOPIC_LABELS` with your chosen names, then re-run the cell.  
Common MD&A topics: _Revenue Growth, Supply Chain, Legal Risk, Liquidity,  
R&D Investment, Macro Outlook, Workforce & Talent, Cybersecurity & Privacy,  
AI & Competition, Shareholder Returns_


In [ ]:
# ── 12.1 Print auto-generated topic names + keywords ─────────────────────────
topic_info_final = topic_model_final.get_topic_info()
print("=== Auto-generated topic names + top 8 keywords ===")
print(f"{'Topic ID':<10} {'Auto Name':<55} Keywords")
print("-" * 110)
for _, row in topic_info_final.iterrows():
    tid = int(row["Topic"])
    if tid == -1:
        continue
    kws = ", ".join(w for w, _ in topic_model_final.get_topic(tid)[:8])
    print(f"{tid:<10} {row['Name']:<55} {kws}")

# ── 12.2 Manual label dict — EDIT THIS ───────────────────────────────────────
# Keys = topic IDs (int). Fill in after inspecting keywords above.
MANUAL_TOPIC_LABELS = {
    # 0:  'Revenue Growth',
    # 1:  'Supply Chain',
    # 2:  'Legal Risk',
    # 3:  'Liquidity & Capital',
    # 4:  'R&D Investment',
    # 5:  'Macro Outlook',
    # 6:  'Workforce & Talent',
    # 7:  'Cybersecurity & Privacy',
    # 8:  'AI & Competition',
    # 9:  'Shareholder Returns',
}

# ── 12.3 Build final label map (fallback to auto-name for unlabelled topics) ──
topic_labels: dict[int, str] = {}
for _, row in topic_info_final.iterrows():
    tid = int(row["Topic"])
    topic_labels[tid] = MANUAL_TOPIC_LABELS.get(tid, row["Name"])

topic_labels[-1] = "Outlier"

print("\n=== Final label map ===")
for tid, lbl in sorted(topic_labels.items()):
    count = topic_info_final.loc[topic_info_final["Topic"] == tid, "Count"].values
    cnt_str = f"{count[0]:,}" if len(count) else "—"
    print(f"  {tid:>4}: {lbl:<45}  ({cnt_str} chunks)")


## 13. Map Topics → Sentences & Export

Maps chunk-level topic assignments back to individual sentences.  
Outputs:

- `datasets/webapp_data/topic_sentences.parquet` — sentence-level topic assignments
- `models/bertopic_final/` — saved BERTopic model (SafeTensors format)


In [ ]:
from pathlib import Path
import pandas as pd

CHUNK_SIZE = 8  # must match Section 3
_META = ["doc_id", "company_name", "filing_type", "filing_date", "year", "quarter"]

# ── 13.1 Reload sentence-level data ──────────────────────────────────────────
_DATA = Path("datasets/final/mda_shared_preprocessed.csv")
sent_df = pd.read_csv(_DATA)

# Assign sentence_id BEFORE any filtering so it matches the FinBERT output
sent_df["sentence_id"] = sent_df.index
sent_df = sent_df[
    sent_df["sentence"].astype(str).str.strip().str.len() > 0
].reset_index(drop=True)

sent_df["sent_idx"] = sent_df.groupby("doc_id").cumcount()
sent_df["chunk_idx"] = sent_df["sent_idx"] // CHUNK_SIZE

print(f"Sentences loaded : {len(sent_df):,}")

# ── 13.2 Rebuild chunk_df to align with topics_final ─────────────────────────
chunk_df = (
    sent_df.groupby(_META + ["chunk_idx"], as_index=False, sort=False)
    .agg(processed_text=("sentence", " ".join))
    .reset_index(drop=True)
)
assert len(chunk_df) == len(topics_final), (
    f"Chunk count mismatch: {len(chunk_df)} vs {len(topics_final)} — "
    "ensure Section 13 runs in the same kernel session as Section 8."
)

chunk_df["topic_id"] = topics_final
chunk_df["topic_label"] = chunk_df["topic_id"].map(topic_labels)
chunk_df["topic_prob"] = [1.0 if t != -1 else 0.0 for t in topics_final]

# ── 13.3 Merge topics back to sentence level ──────────────────────────────────
topic_sent_df = sent_df.merge(
    chunk_df[_META + ["chunk_idx", "topic_id", "topic_label", "topic_prob"]],
    on=_META + ["chunk_idx"],
    how="left",
)

out_df = (
    topic_sent_df[
        [
            "sentence_id",
            "doc_id",
            "company_name",
            "filing_type",
            "year",
            "quarter",
            "topic_id",
            "topic_label",
            "topic_prob",
        ]
    ]
    .rename(columns={"company_name": "ticker"})
    .copy()
)

print(f"Total sentences   : {len(out_df):,}")
print(f"Outliers (id=-1)  : {(out_df['topic_id'] == -1).sum():,}")
print("\nTopic distribution:")
print(
    out_df.groupby(["topic_id", "topic_label"])
    .size()
    .reset_index(name="sentences")
    .to_string(index=False)
)

# ── 13.4 Save parquet ─────────────────────────────────────────────────────────
OUT_DIR = Path("datasets/webapp_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "topic_sentences.parquet"
out_df.to_parquet(OUT_PATH, index=False)
print(f"\n✅ Saved {len(out_df):,} rows → {OUT_PATH}")

# ── 13.5 Save BERTopic model ──────────────────────────────────────────────────
MODEL_DIR = Path("models/bertopic_final")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
topic_model_final.save(
    str(MODEL_DIR),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBED_MODEL,
)
print(f"✅ BERTopic model saved → {MODEL_DIR}")


## 14. Random Search for Outlier Reduction (BERTopic)

Use this as an alternative to full grid search when outliers are high.

What this does:

- Randomly samples BERTopic/HDBSCAN/vectorizer settings
- Optionally applies `reduce_outliers(..., strategy='embeddings')`
- Ranks trials by a score that prioritizes low outlier rate while discouraging extreme topic counts

Tip:

- Start with `n_iter=10` for a quick pass
- Increase to `20-40` for a more stable search


In [ ]:
# Random-search utility for BERTopic outlier tuning
from sklearn.model_selection import ParameterSampler


def run_bertopic_random_search(
    n_iter=12,
    random_state=42,
    apply_reduce_outliers=True,
    target_topics=(6, 20),
):
    required_names = [
        "docs",
        "embeddings",
        "embed_model",
        "umap_embeddings",
        "USE_CUML",
        "GpuHDBSCAN",
        "CpuHDBSCAN",
        "PassthroughUMAP",
        "ClassTfidfTransformer",
        "BERTopic",
    ]
    missing = [n for n in required_names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required objects in current kernel: "
            + ", ".join(missing)
            + ". Re-run earlier BERTopic setup/training sections first."
        )

    param_space = {
        "min_cluster_size": [15, 20, 30, 40, 50, 75, 100],
        "min_samples_ratio": [0.15, 0.20, 0.25, 0.30],
        "min_df": [3, 5, 8, 12],
        "ngram_upper": [1, 2],
        "outlier_threshold": [0.0, 0.05, 0.10, 0.15],
    }

    sampler = list(
        ParameterSampler(param_space, n_iter=n_iter, random_state=random_state)
    )
    n_total = len(docs)
    results = []

    print(f"Running random search with {len(sampler)} trials...")

    for i, cfg in enumerate(sampler, 1):
        mcs = int(cfg["min_cluster_size"])
        min_samples = max(5, int(round(mcs * float(cfg["min_samples_ratio"]))))

        if USE_CUML:
            hdb_exp = GpuHDBSCAN(
                min_cluster_size=mcs,
                min_samples=min_samples,
                metric="euclidean",
                prediction_data=True,
            )
        else:
            hdb_exp = CpuHDBSCAN(
                min_cluster_size=mcs,
                min_samples=min_samples,
                metric="euclidean",
                prediction_data=True,
            )

        vec_exp = CountVectorizer(
            stop_words="english",
            max_df=0.95,
            min_df=int(cfg["min_df"]),
            ngram_range=(1, int(cfg["ngram_upper"])),
        )

        model_exp = BERTopic(
            embedding_model=embed_model,
            umap_model=PassthroughUMAP(umap_embeddings),
            hdbscan_model=hdb_exp,
            vectorizer_model=vec_exp,
            ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
            calculate_probabilities=False,
            verbose=False,
        )

        topics_raw, _ = model_exp.fit_transform(docs, embeddings)

        if apply_reduce_outliers:
            topics_use = model_exp.reduce_outliers(
                docs,
                topics_raw,
                strategy="embeddings",
                embeddings=embeddings,
                threshold=float(cfg["outlier_threshold"]),
            )
            model_exp.update_topics(docs, topics=topics_use)
        else:
            topics_use = topics_raw

        n_outliers = int(sum(1 for t in topics_use if t == -1))
        outlier_pct = 100 * n_outliers / n_total
        n_topics = int(len(set(t for t in topics_use if t != -1)))

        lo, hi = target_topics
        topic_penalty = 0.0 if lo <= n_topics <= hi else abs(n_topics - (lo + hi) / 2)

        # Higher score is better: low outlier % and reasonable topic count
        score = -(outlier_pct + 1.25 * topic_penalty)

        results.append(
            {
                "trial": i,
                "score": score,
                "outlier_pct": outlier_pct,
                "n_topics": n_topics,
                "n_outliers": n_outliers,
                "min_cluster_size": mcs,
                "min_samples": min_samples,
                "min_df": int(cfg["min_df"]),
                "ngram_upper": int(cfg["ngram_upper"]),
                "outlier_threshold": float(cfg["outlier_threshold"]),
                "model": model_exp,
                "topics": topics_use,
            }
        )

        print(
            f"[{i:>2}/{len(sampler)}] mcs={mcs:>3} ms={min_samples:>2} "
            f"min_df={int(cfg['min_df']):>2} ngram<= {int(cfg['ngram_upper'])} "
            f"thr={float(cfg['outlier_threshold']):.2f} | "
            f"topics={n_topics:>3} outliers={outlier_pct:>5.1f}% score={score:>7.2f}"
        )

    res_df = (
        pd.DataFrame(results)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

    cols = [
        "trial",
        "score",
        "outlier_pct",
        "n_topics",
        "n_outliers",
        "min_cluster_size",
        "min_samples",
        "min_df",
        "ngram_upper",
        "outlier_threshold",
    ]

    print("\nTop random-search trials:")
    display(res_df[cols].head(10))

    return res_df


In [ ]:
# Example run + pick best model
RUN_RANDOM_SEARCH = False  # set True to execute

if RUN_RANDOM_SEARCH:
    rs_results = run_bertopic_random_search(
        n_iter=12,
        random_state=42,
        apply_reduce_outliers=True,
        target_topics=(6, 20),
    )

    best = rs_results.iloc[0]
    topic_model_random_best = best["model"]
    topics_random_best = best["topics"]

    print("\nBest random-search configuration:")
    print(
        best[
            [
                "score",
                "outlier_pct",
                "n_topics",
                "n_outliers",
                "min_cluster_size",
                "min_samples",
                "min_df",
                "ngram_upper",
                "outlier_threshold",
            ]
        ]
    )

    # Optional: compare to current final model in one line
    curr_outlier_pct = 100 * sum(1 for t in topics_final if t == -1) / len(topics_final)
    new_outlier_pct = float(best["outlier_pct"])
    print(f"\nCurrent final outlier%: {curr_outlier_pct:.2f}%")
    print(f"Random-search best outlier%: {new_outlier_pct:.2f}%")
else:
    print("Set RUN_RANDOM_SEARCH = True to run random search.")